# 📱 Notebook 5: Genie Mobile — Setup, Configuration & Demo Guide

## Genie Family PoC Workshop | Manufacturing Industry

**Purpose**: Guide IT/security teams through mobile app deployment and provide a step-by-step demo script for business users.

**Target personas**: IT Admin (setup), Executive/Business User (demo)

**What this notebook covers**:
1. Account-level configuration and prerequisites
2. Security checklist with automated validation
3. MDM deployment guide (iOS & Android)
4. Step-by-step demo script for customer walkthroughs
5. Troubleshooting guide

**Runtime**: ~5 minutes (configuration) + guided walkthrough

---

### 📱 What is Genie Mobile?

The Databricks Genie mobile app is a **native iOS and Android** client for Genie One. It provides:
- Natural-language data questions
- Mobile-optimized dashboard interaction
- Dashboard chat (Ask Genie within a dashboard)
- Databricks Apps
- Scheduled task creation and push notifications
- Workspace switching

**Key security facts**:
- Same OAuth flow as web (no separate IdP registration needed)
- Unity Catalog enforced — same row/column security
- No mobile-only backend or data store
- Tokens in device encrypted secure store
- Supports compliance security profile workspaces

**Status**: Public Preview (enabled by default)


## 🔧 Cell 1: Prerequisites Check

This cell validates that your workspace is ready for Genie Mobile deployment.


In [ ]:
# Prerequisites validation
import requests

WORKSPACE_URL = spark.conf.get("spark.databricks.workspaceUrl")
TOKEN = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()

headers = {"Authorization": f"Bearer {TOKEN}", "Content-Type": "application/json"}

print("📱 GENIE MOBILE — PREREQUISITES CHECK")
print("=" * 60)

# Check 1: Workspace URL
print(f"\n1️⃣ Workspace URL: https://{WORKSPACE_URL}")
print(f"   ✅ Accessible")

# Check 2: SQL Warehouses
wh_resp = requests.get(f"https://{WORKSPACE_URL}/api/2.0/sql/warehouses", headers=headers)
if wh_resp.status_code == 200:
    warehouses = wh_resp.json().get("warehouses", [])
    serverless = [w for w in warehouses if w.get("enable_serverless_compute") or w.get("warehouse_type") == "PRO"]
    if serverless:
        print(f"   ✅ Serverless SQL Warehouse: {serverless[0]['name']}")
    else:
        print(f"   ⚠️ No Serverless SQL Warehouse found — required for Genie")

# Check 3: Genie Agent exists
print(f"\n2️⃣ Genie Agent (from Notebook 2):")
print(f"   💡 Verify the Manufacturing Operations Intelligence agent exists")
print(f"   🔗 Check: https://{WORKSPACE_URL}/genie")

# Check 4: Mobile-specific
print(f"\n3️⃣ Mobile-Specific Requirements:")
print(f"   📋 Account Console → Previews → 'Genie One Mobile' → Enabled")
print(f"   📋 Partner-Powered AI → Enabled")
print(f"   📋 SSO/MFA configured in your IdP")
print(f"   📋 IP Access List includes mobile VPN egress IPs (if applicable)")

print(f"\n{'=' * 60}")
print(f"✅ Workspace checks complete. Review any ⚠️ items above.")


## 🔒 Cell 2: Security Configuration Guide

This section covers the security setup for IT/security teams deploying the Genie mobile app.

### Authentication Flow
```
User opens Genie app
    → App opens system browser
    → Browser navigates to login.databricks.com
    → IdP SSO/MFA flow (your existing policies apply)
    → OAuth token returned to app
    → App stores token in device encrypted keychain
    → All subsequent requests use this token
```

### Security Architecture
| Layer | Control | Details |
|---|---|---|
| **Identity** | SSO + MFA | Your existing IdP policies apply automatically |
| **Network** | IP Access List / VPN | Add mobile VPN egress IPs to workspace IP ACL |
| **Data** | Unity Catalog | Row filters, column masks, table permissions — all enforced |
| **Device** | MDM + Device Posture | Deploy via MDM, enforce device compliance |
| **Token** | Encrypted Keychain | OAuth tokens in OS secure store, auto-rotation |
| **Compute** | Workspace | All queries run in your workspace — app only renders results |


In [ ]:
# Security checklist for IT teams
security_checklist = [
    ("Identity & Access", [
        ("SSO configured for workspace", "Account Console → Settings → SSO"),
        ("MFA enabled in IdP", "Your IdP admin console"),
        ("Conditional access policies include mobile", "Your IdP admin console"),
        ("Consumer access group created", "Workspace → Admin → Groups"),
        ("Users assigned to consumer group", "Workspace → Admin → Users"),
    ]),
    ("Network Security", [
        ("IP Access List configured", "Workspace → Settings → IP Access Lists"),
        ("Mobile VPN egress IPs added to ACL", "Add your VPN provider's egress IPs"),
        ("login.databricks.com reachable from mobile", "Test from a mobile device on VPN"),
        ("Workspace URL reachable from mobile", "Test from a mobile device on VPN"),
    ]),
    ("Mobile Device Management", [
        ("MDM platform selected", "Intune, Workspace ONE, Jamf, etc."),
        ("App ID registered: com.databricks.one.mobile", "Add to MDM app catalog"),
        ("Per-app VPN configured (if required)", "MDM → VPN → Per-app rules"),
        ("Device compliance policy set", "MDM → Compliance → Require encryption, PIN"),
    ]),
    ("Databricks Configuration", [
        ("Genie One Mobile preview enabled", "Account Console → Previews"),
        ("Partner-Powered AI enabled", "Account Console → Settings"),
        ("Serverless SQL Warehouse provisioned", "Workspace → SQL Warehouses"),
        ("Genie Agent shared with target users", "Genie Agent → Share → Add users/groups"),
    ]),
]

print("🔒 GENIE MOBILE — SECURITY DEPLOYMENT CHECKLIST")
print("=" * 60)
for category, items in security_checklist:
    print(f"\n📋 {category}:")
    for item, location in items:
        print(f"   ☐ {item}")
        print(f"     → {location}")
print(f"\n{'=' * 60}")
print("Print this checklist and work through it with the customer's IT team.")


## 📱 Cell 3: Mobile Demo Script

This is the **step-by-step demo script** for showing Genie Mobile to customers. Each step includes what to do, what to say, and what to highlight.

### Pre-Demo Setup
1. Install the Genie app on your phone (iOS: App Store, Android: Google Play)
2. Sign in to the customer's workspace (or your demo workspace)
3. Verify the Manufacturing Operations Intelligence agent is accessible
4. Have a dashboard ready (created in Notebook 4 or manually)

### Demo Duration: 10-15 minutes


In [ ]:
# Mobile demo script
demo_script = [
    {
        "step": 1,
        "action": "Install & Sign In",
        "what_to_do": "Open the app, tap 'Log in', complete SSO flow",
        "what_to_say": "The app uses your existing SSO — same credentials as the web. No separate registration needed.",
        "what_to_highlight": "SSO flow, MFA if enabled, workspace selection"
    },
    {
        "step": 2,
        "action": "Browse Homepage",
        "what_to_do": "Show the Genie One homepage with pinned assets",
        "what_to_say": "This is the same Genie One experience as the web, optimized for mobile. You can see pinned dashboards, agents, and apps.",
        "what_to_highlight": "Pinned assets, search bar, For You recommendations"
    },
    {
        "step": 3,
        "action": "Ask a Question",
        "what_to_do": "Type: 'What was our yield rate across all plants this week?'",
        "what_to_say": "I can ask any question in natural language. Genie routes it to the right agent and generates the answer.",
        "what_to_highlight": "Natural language input, agent routing, SQL generation"
    },
    {
        "step": 4,
        "action": "Follow-Up Question",
        "what_to_do": "Type: 'Which plant had the lowest yield?'",
        "what_to_say": "I can drill deeper with follow-up questions. Genie remembers the context from my previous question.",
        "what_to_highlight": "Context retention, conversational flow"
    },
    {
        "step": 5,
        "action": "Open a Dashboard",
        "what_to_do": "Navigate to a pinned dashboard, interact with filters",
        "what_to_say": "Dashboards are fully interactive on mobile — filters, drill-downs, and cross-filtering all work.",
        "what_to_highlight": "Mobile-optimized layout, touch interactions, filter controls"
    },
    {
        "step": 6,
        "action": "Chat with Dashboard",
        "what_to_do": "Tap 'Ask Genie' within the dashboard, ask: 'Why did the Detroit plant decline?'",
        "what_to_say": "I can ask follow-up questions directly within a dashboard. Genie uses the dashboard context to give more relevant answers.",
        "what_to_highlight": "Dashboard chat, contextual analysis"
    },
    {
        "step": 7,
        "action": "Create Scheduled Task",
        "what_to_do": "Create a task: 'Every Monday at 7am, summarize last week's KPIs and flag underperforming plants'",
        "what_to_say": "I can set up automated reports right from my phone. Every Monday morning, I'll get a push notification with the results.",
        "what_to_highlight": "Scheduled task creation, push notifications (iOS)"
    },
    {
        "step": 8,
        "action": "Switch Workspace",
        "what_to_do": "Show workspace switcher (if multiple workspaces available)",
        "what_to_say": "If you have multiple workspaces — say, one for production and one for development — you can switch without signing out.",
        "what_to_highlight": "Multi-workspace support"
    },
    {
        "step": 9,
        "action": "Security Discussion",
        "what_to_do": "Show that restricted data is not visible (if RLS configured)",
        "what_to_say": "Everything you see here respects Unity Catalog permissions. Row-level security, column masking — it all applies on mobile exactly as on the web.",
        "what_to_highlight": "UC governance, same permissions as web"
    },
    {
        "step": 10,
        "action": "Closing",
        "what_to_do": "Summarize the experience",
        "what_to_say": "Your plant managers, executives, and field teams can now access governed data insights from anywhere — no laptop required. And it's free through January 2027.",
        "what_to_highlight": "Anywhere access, governed, free promotion"
    },
]

print("📱 GENIE MOBILE — DEMO SCRIPT")
print("=" * 60)
for step in demo_script:
    print(f"\n{'─' * 60}")
    print(f"Step {step['step']}: {step['action']}")
    print(f"{'─' * 60}")
    print(f"  🎬 DO:        {step['what_to_do']}")
    print(f"  🗣️ SAY:       {step['what_to_say'][:100]}...")
    print(f"  ⭐ HIGHLIGHT: {step['what_to_highlight']}")
print(f"\n{'=' * 60}")
print("\n📥 App Download Links:")
print("   iOS:     https://apps.apple.com/app/databricks-genie/id6758961415")
print("   Android: https://play.google.com/store/apps/details?id=com.databricks.one.mobile")


## 🔧 Cell 4: Troubleshooting Guide

Common issues and resolutions for the Genie mobile app.


In [ ]:
# Troubleshooting guide
troubleshooting = [
    {
        "issue": "Email verification failed / OTP not received",
        "cause": "Account-level sign-in requires email OTP that may go to an inaccessible mailbox",
        "solution": "Use 'Enter workspace link' sign-in method instead. Enter your workspace URL directly to authenticate via workspace SSO."
    },
    {
        "issue": "Cannot connect to workspace",
        "cause": "Network restrictions blocking mobile access",
        "solution": "1. Verify VPN is connected\n2. Check IP Access List includes VPN egress IPs\n3. Ensure login.databricks.com is reachable\n4. Test workspace URL in mobile browser first"
    },
    {
        "issue": "No assets visible after sign-in",
        "cause": "User lacks workspace entitlements",
        "solution": "Grant at least Consumer access to the user. Workspace → Admin → Users → Select user → Add 'Consumer access' entitlement."
    },
    {
        "issue": "Push notifications not working",
        "cause": "iOS notification permissions not granted",
        "solution": "iOS Settings → Notifications → Genie One → Allow Notifications = ON. Note: Push notifications are iOS only currently."
    },
    {
        "issue": "Dashboard not loading or slow",
        "cause": "SQL warehouse not running or undersized",
        "solution": "1. Check warehouse status in workspace\n2. Ensure auto-start is enabled\n3. Consider increasing warehouse size for mobile users"
    },
    {
        "issue": "MCP connections not available",
        "cause": "MCP connections must be configured on web first",
        "solution": "Configure MCP connections in Genie One web experience. They are automatically inherited by the mobile app."
    },
]

print("🔧 GENIE MOBILE — TROUBLESHOOTING GUIDE")
print("=" * 60)
for i, item in enumerate(troubleshooting, 1):
    print(f"\n{'─' * 60}")
    print(f"Issue {i}: {item['issue']}")
    print(f"{'─' * 60}")
    print(f"  Cause:    {item['cause']}")
    print(f"  Solution: {item['solution']}")
print(f"\n{'=' * 60}")


## 🎉 Summary — Complete PoC Workshop

### 🏁 You've completed all 5 notebooks!

| Notebook | What Was Built | Key Artifact |
|---|---|---|
| **1. Foundation** | 10 synthetic manufacturing tables + metric views | Unity Catalog schema with rich comments |
| **2. Genie Agent** | Fully curated agent with instructions, queries, benchmarks | Manufacturing Operations Intelligence agent |
| **3. Genie One** | Homepage config, demo script, scheduled tasks | Business user enablement package |
| **4. Genie Code** | Chat mode, agent mode, dashboard authoring demos | Technical practitioner demo kit |
| **5. Genie Mobile** | Security checklist, MDM guide, demo script | IT deployment + mobile demo package |

### 📊 PoC Success Metrics
Track these during the PoC to measure success:

| Metric | Target | How to Measure |
|---|---|---|
| Agent accuracy | >85% | Benchmark questions pass rate |
| User adoption | >50% of invited users | Genie One active users |
| Time to insight | <30 seconds | Average question-to-answer time |
| User satisfaction | >4/5 | Feedback survey |
| Questions answered | >80% | Unanswered question rate |

### 🔗 Key Resources
- [Genie Documentation](https://docs.databricks.com/aws/en/genie)
- [Genie Mobile Guide](https://docs.databricks.com/aws/en/genie-one/mobile)
- [Genie Code Guide](https://docs.databricks.com/aws/en/genie-code)
- [Genie Agents Guide](https://docs.databricks.com/aws/en/genie-agents)
